# Alterei em 17/07/2024 o script para usar a data de modificação do arquivo ao inves da data de criação do arquivo

# Setores Novos 4G
### Essa versao considera o ano de criação do arquivo para evitar problemas na virada de ano.

In [ ]:
import os
import re
import pandas as pd
from datetime import datetime

def get_file_info(filename, directory):
    match = re.search(r'_(\d+)\.txt', filename)
    week_number = int(match.group(1)) if match else None

    file_path = os.path.join(directory, filename)
    modification_date = datetime.fromtimestamp(os.path.getmtime(file_path)).year

    return modification_date, week_number

def get_latest_files(directory, include_latest=False, num_files=4):
    all_files = [f for f in os.listdir(directory) if re.match(r'EUtranCell_TDD_FDD_\d+\.txt', f)]
    sorted_files = sorted(all_files, key=lambda f: get_file_info(f, directory), reverse=True)
    
    if include_latest:
        return sorted_files[:num_files]  # Inclui a semana mais recente
    else:
        return sorted_files[1:num_files+1]  # Exclui a semana mais recente

def concatenate_files_to_df(directory, files):
    df_list = []
    for file in files:
        year, week_number = get_file_info(file, directory)
        file_path = os.path.join(directory, file)
        df = pd.read_csv(file_path, sep='\t', encoding='ISO-8859-1', usecols=['UF', 'MUNICIPIO', 'eNB', 'CELL', 'earfcndl', 'CGI', 'operationalState'])
        df['Semana'] = week_number
        df['Ano'] = year
        df_list.append(df)
    return pd.concat(df_list, ignore_index=True)

# Caminho do diretório
directory = 'C:/Camilo/'

# Obter os 4 arquivos das últimas semanas, excluindo a semana mais recente
previous_weeks_files = get_latest_files(directory)
EUtranCell = concatenate_files_to_df(directory, previous_weeks_files)

EUtranCell = EUtranCell[EUtranCell['operationalState'] == 'ENABLED']
EUtranCell.drop(columns=['operationalState', 'Ano','Semana', 'MUNICIPIO'], inplace=True)
EUtranCell.drop_duplicates(subset=['UF', 'eNB', 'CELL', 'earfcndl', 'CGI'], inplace=True)

# Carregar os dados da semana mais recente
semana_atual_files = get_latest_files(directory, include_latest=True, num_files=1)
semana_atual = concatenate_files_to_df(directory, semana_atual_files)

semana_atual = semana_atual[semana_atual['operationalState'] == 'ENABLED']
semana_atual.drop(columns='operationalState', inplace=True)

# Comparar os dados da semana mais recente com os dados anteriores
setores_novos = pd.merge(semana_atual, EUtranCell, on=['UF', 'eNB', 'CELL', 'earfcndl', 'CGI'], how='left', indicator=True)
setores_novos = setores_novos[setores_novos['_merge'] == 'left_only']

# Identificar novos eNBs e novos Setores
enbs_unicos = set(EUtranCell['eNB'].unique())
setores_novos['Novo'] = setores_novos.apply(lambda row: 'Novo eNB' if row['eNB'] not in enbs_unicos else 'Novo Setor', axis=1)

# Remover a coluna '_merge', pois não é mais necessária
setores_novos.drop(columns=['_merge'], inplace=True)

# Substituir os valores de 'UF' por 'Regional'
setores_novos['Regional'] = setores_novos['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], 
                                                        ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])

# Reorganizar as colunas
regional_col = setores_novos.pop('Regional')
setores_novos.insert(0, 'Regional', regional_col)
setores_novos.sort_values(by=['Regional', 'UF', 'eNB', 'CELL'], ascending=[True, True, True, True], inplace=True)

# Exibir a forma do DataFrame resultante
setores_novos.head()

# Setores Novos 5G

In [ ]:
import os
import re
import pandas as pd
from datetime import datetime

def get_file_info(filename, directory):
    match = re.search(r'_(\d+)\.txt', filename)
    week_number = int(match.group(1)) if match else None

    file_path = os.path.join(directory, filename)
    modification_date = datetime.fromtimestamp(os.path.getmtime(file_path)).year

    return modification_date, week_number

def get_latest_files(directory, include_latest=False, num_files=4):
    all_files = [f for f in os.listdir(directory) if re.match(r'5G_NRCellDU_\d+\.txt', f)]
    sorted_files = sorted(all_files, key=lambda f: get_file_info(f, directory), reverse=True)
    if include_latest:
        return sorted_files[:num_files]  # Inclui a semana mais recente
    else:
        return sorted_files[1:num_files+1]  # Exclui a semana mais recente

def concatenate_files_to_df(directory, files):
    df_list = []
    for file in files:
        year, week_number = get_file_info(file, directory)
        file_path = os.path.join(directory, file)
        df = pd.read_csv(file_path, sep='\t', encoding='ISO-8859-1', usecols=['UF', 'eNB', 'nRCellDUId', 'operationalState'])
        df['Ano'] = year
        df['Semana'] = week_number
        df_list.append(df)
    return pd.concat(df_list, ignore_index=True)

# Caminho do diretório
directory = 'C:/Camilo'

# Obter os 4 arquivos das últimas semanas, excluindo a semana mais recente
previous_weeks_files = get_latest_files(directory)
NRCell = concatenate_files_to_df(directory, previous_weeks_files)

NRCell = NRCell[NRCell['operationalState'] == 'ENABLED']
NRCell.drop(columns=['operationalState', 'Ano', 'Semana'], inplace=True)
NRCell.drop_duplicates(subset=['UF', 'eNB', 'nRCellDUId'], inplace=True)

# Carregar os dados da semana mais recente
semana_atual_files = get_latest_files(directory, include_latest=True, num_files=1)
semana_atual = concatenate_files_to_df(directory, semana_atual_files)

semana_atual = semana_atual[semana_atual['operationalState'] == 'ENABLED']
semana_atual.drop(columns='operationalState', inplace=True)

# Comparar os dados da semana mais recente com os dados anteriores
setores_novos_NR = pd.merge(semana_atual, NRCell, on=['UF', 'eNB', 'nRCellDUId'], how='left', indicator=True)
setores_novos_NR = setores_novos_NR[setores_novos_NR['_merge'] == 'left_only']

# Identificar novos eNBs e novos Setores
enbs_unicos = set(NRCell['eNB'].unique())
setores_novos_NR['Novo'] = setores_novos_NR.apply(lambda row: 'Novo eNB' if row['eNB'] not in enbs_unicos else 'Novo Setor', axis=1)
setores_novos_NR.rename(columns={'eNB': 'gNB'}, inplace=True)

# Remover a coluna '_merge', pois não é mais necessária
setores_novos_NR.drop(columns=['_merge'], inplace=True)

# Substituir os valores de 'UF' por 'Regional'
setores_novos_NR['Regional'] = setores_novos_NR['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], 
                                                             ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])

# Usar pop para remover a coluna e depois inseri-la no início
regional_col = setores_novos_NR.pop('Regional')
setores_novos_NR.insert(0, 'Regional', regional_col)
setores_novos_NR.sort_values(by=['Regional', 'UF', 'gNB', 'nRCellDUId'], ascending=[True, True, True, True], inplace=True)

# Exibir as primeiras linhas do DataFrame resultante
setores_novos_NR.head()


In [ ]:
from datetime import datetime
import pandas as pd

semana = datetime.today().strftime('%U')
semana_atual = int(semana) + 1

arquivo_excel = 'C:/Vivo Reports/Sites Novos/' + 'Sites_Novos' + '_' + str(semana_atual) + '.xlsx'

with pd.ExcelWriter(arquivo_excel, engine='openpyxl') as writer:
    setores_novos.to_excel(writer, sheet_name='Setores Novos LTE', index=False)
    setores_novos_NR.to_excel(writer, sheet_name='Setores Novos NR', index=False)

print('Sites Novos, Done!')

# Historico Sites Novos
### O objetivo é obter o histórico de implementações da rede.

In [ ]:
import pandas as pd
import glob

# Caminho para a pasta com os arquivos .xlsx
caminho = r'C:/Vivo Reports/Sites Novos/*.xlsx'

# Criar listas vazias para armazenar os dados de cada aba
dados_lte = []
dados_nr = []

# Iterar sobre cada arquivo na pasta
for arquivo in glob.glob(caminho):
    # Ler a aba "Setores Novos LTE"
    df_lte = pd.read_excel(arquivo, sheet_name='Setores Novos LTE', engine='openpyxl')
    dados_lte.append(df_lte)  # Adiciona o DataFrame à lista
    
    # Ler a aba "Setores Novos NR"
    df_nr = pd.read_excel(arquivo, sheet_name='Setores Novos NR', engine='openpyxl')
    dados_nr.append(df_nr)  # Adiciona o DataFrame à lista

# Concatenar todos os dataframes de cada lista para formar os dataframes finais
df_historico_LTE = pd.concat(dados_lte, ignore_index=True)
df_historico_NR = pd.concat(dados_nr, ignore_index=True)

# Ordenando o historico do mais antigo para o mais atual 
df_historico_LTE.drop_duplicates(inplace=True)

df_historico_LTE.sort_values(by=['Ano', 'Semana','Regional', 'UF', 'eNB', 'CELL'], ascending=[True, True, True, True, True, True],inplace=True)

# Ordenando o historico do mais antigo para o mais atual
df_historico_NR.drop_duplicates(inplace=True)

df_historico_NR.sort_values(by=['Ano', 'Semana','Regional', 'UF', 'gNB', 'nRCellDUId'], ascending=[True, True, True, True, True, True],inplace=True)

# Caminho onde o arquivo Excel será salvo
caminho_salvar = r'C:\Vivo Reports\Historico Ativação\Historico_Ativação.xlsx'

# Usar ExcelWriter para escrever em abas diferentes no mesmo arquivo
with pd.ExcelWriter(caminho_salvar, engine='openpyxl') as writer:
    df_historico_LTE.to_excel(writer, sheet_name='Historico_LTE', index=False)
    df_historico_NR.to_excel(writer, sheet_name='Historico_NR', index=False)

print('Historico_Ativação, Done!')

# Atualizar o arquivo Historico com base apenas no ultimo arquivo de Sites Novos
## Testar essa solução a partir da proxima semana

In [ ]:
# O historico completo, incluindo a semana atual, foi reconstruido na celula anterior.
# Nao acrescentar novamente o arquivo mais recente.
